# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [1]:
from datetime import datetime
from pathlib import Path

from IPython.display import HTML, display

from modules.extraccion import (
    COLUMNAS_DISTRIBUCIONES,
    URL_PAGINA,
    obtener_distribuciones,
    obtener_tabla_fibras_en_notebook,
)
from modules.presentacion import (
    crear_ficha_rendimiento,
    exportar_csv_analitico,
    exportar_csv_excel,
    exportar_xlsx,
)

In [2]:
CARPETA_SALIDA = Path.cwd() / "output"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Ejecutar extracción

In [3]:
print(f"Consultando {URL_PAGINA} ...")
df = obtener_tabla_fibras_en_notebook(headless=HEADLESS, timeout_datos_ms=TIMEOUT_DATOS_MS)
print(f"Índice FIBRAS - {datetime.now():%Y-%m-%d %H:%M} (dato con ~20 min de retraso)")
display(df)

if EXPORTAR_CSV_ANALITICO:
    ruta_csv_analitico = exportar_csv_analitico(df, CARPETA_SALIDA)
    print(f"CSV analítico guardado en: {ruta_csv_analitico}")

Consultando https://amefibra.com/el-mercado/indice-fibras/ ...


Aviso: se agotó el tiempo esperando datos en vivo; se usará lo cargado.


Índice FIBRAS - 2026-08-22 18:45 (dato con ~20 min de retraso)


,Emisora,Cotización,Var.,Var. %,Apertura,Máx. día,Min. día,Promedio,Operaciones,Volumen,Importe,Máx. 52 s.,Min. 52 s.
0,DANHOS13,28.71,0.10,0.35%,28.77,28.75,29.00,28.54,2262,164212,4717301,29.12,23.65
1,EDUCA18,52.50,-1.50,-2.78%,52.50,52.50,52.50,52.50,11,366,19215,58.36,46.39
2,FIBRAMQ12,43.51,0.76,1.78%,43.05,42.95,43.95,42.16,1536,890429,38525363,45.27,27.73
3,FIBRAPL14,75.57,1.07,1.44%,75.43,74.80,76.06,74.80,4503,513731,38792854,83.97,64.05
4,FIBRAUP18,37.45,0.00,0.00%,37.45,37.45,37.45,37.45,11,32,1194,41.00,17.27
5,FIHO12,7.65,0.11,1.46%,7.59,7.65,7.66,7.53,236,23192,177352,8.01,6.92
6,FINN13,4.86,0.09,1.89%,4.83,4.77,4.88,4.77,179,8250,40058,5.40,4.33
7,FMTY14,14.50,0.28,1.97%,14.43,14.27,14.59,14.26,28161,7460899,107779291,15.79,12.32
8,FNOVA17,41.59,-0.13,-0.31%,42.16,42.01,42.80,41.52,369,13733,572960,45.95,27.00
9,FPLUS16,5.09,0.02,0.39%,5.05,5.10,5.10,5.01,130,10960,55246,6.00,4.82


CSV analítico guardado en: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260822_184555_indice_fibras_amefibra.csv


### Exportar a archivo de Excel - xlsx (Ejecución opcional)

In [4]:
if EXPORTAR_CSV_EXCEL:
    ruta_csv_excel = exportar_csv_excel(df, RUTA_CSV_EXCEL)
    print(f"CSV compatible con Excel guardado en: {ruta_csv_excel}")

if EXPORTAR_XLSX:
    ruta_xlsx = exportar_xlsx(df, RUTA_XLSX)
    print(f"Excel guardado en: {ruta_xlsx}")

## Solo emisoras

In [5]:
df_emisoras = df[["Emisora"]]
print(df_emisoras)

      Emisora
0    DANHOS13
1     EDUCA18
2   FIBRAMQ12
3   FIBRAPL14
4   FIBRAUP18
5      FIHO12
6      FINN13
7      FMTY14
8     FNOVA17
9     FPLUS16
10    FSHOP13
11     FUNO11
12     NEXT25
13     SOMA21
14  STORAGE18


## Historial de distribuciones por FIBRA

### Fuentes evaluadas

| Fuente | Cobertura BMV | Datos de distribuciones | Acceso y límites |
|---|---|---|---|
| Relación con Inversionistas del emisor | Sí, por emisora | Fuente primaria; puede incluir fechas, importe y componentes fiscales en PDF/XLSX | Gratuita, sin API uniforme; requiere localizar y procesar reportes de cada emisor |
| AMEFIBRA | Sí, índice agregado | Cotización e indicadores del índice; no publica aquí un histórico normalizado de distribuciones | Consulta web pública; no se expone una API de dividendos en esta tabla |
| BMV/BIVA | Sí | Información oficial de emisoras y eventos, según disponibilidad del portal | Consulta pública, pero sin una API gratuita y estable para este flujo |
| FMP, Alpha Vantage, Twelve Data, EODHD, Nasdaq Data Link y Polygon | Cobertura mexicana variable | La cobertura y profundidad de dividendos para tickers BMV no está garantizada en el plan gratuito | Requieren revisar ticker, API key y límites por proveedor |
| `yfinance` | Sí para tickers Yahoo con sufijo `.MX`, cuando Yahoo dispone del evento | Fecha ex-dividendo y monto; no garantiza fecha de registro, pago ni componentes fiscales | Gratis y sin API key, pero es un cliente no oficial de Yahoo Finance y está sujeto a cambios y límites |

Se usa `yfinance` como respaldo reproducible porque las fuentes primarias no ofrecen una API homogénea. El resultado contiene la fecha ex-dividendo y el importe disponible en Yahoo; la fecha de registro, fecha de pago y componentes fiscales no se incluyen porque esta fuente no los entrega de forma confiable. El histórico se ordena del más antiguo al más reciente. `yield_pct` es el rendimiento de cada distribución respecto al cierre de su fecha ex-dividendo; `annualized_yield_pct` anualiza ese rendimiento usando `365 / días_del_periodo`. Para la primera fila se usa la mediana histórica de días entre distribuciones.

In [6]:
TICKER_PRUEBA = "FUNO11"
assert TICKER_PRUEBA in set(df_emisoras["Emisora"]), "El ticker de prueba no está en el listado AMEFIBRA."

historial_dividendos = obtener_distribuciones(TICKER_PRUEBA, CARPETA_SALIDA)
assert list(historial_dividendos.columns) == COLUMNAS_DISTRIBUCIONES
assert historial_dividendos["ex_date"].is_monotonic_increasing
assert not historial_dividendos.duplicated(subset=["ticker", "ex_date", "amount_mxn"]).any()
assert (historial_dividendos["amount_mxn"] > 0).all()
assert historial_dividendos["annualized_yield_pct"].notna().all()
assert Path(historial_dividendos.attrs["ruta_csv"]).exists()

print(f"Ticker probado: {TICKER_PRUEBA}. Registros: {len(historial_dividendos)}")
print(f"Periodicidad detectada: {historial_dividendos['periodicity'].iloc[0]}")
print(f"CSV generado: {historial_dividendos.attrs['ruta_csv']}")
display(historial_dividendos.tail(10))

Ticker probado: FUNO11. Registros: 64
Periodicidad detectada: trimestral
CSV generado: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260822_184557_FUNO11_dividendos.csv


,ticker,ex_date,amount_mxn,close_on_ex_date_mxn,yield_pct,annualized_yield_pct,periodicity
54,FUNO11,2024-05-07,0.402963,25.280001,1.593999,10.578358,trimestral
55,FUNO11,2024-08-08,0.519023,24.420000,2.125401,8.341629,trimestral
56,FUNO11,2024-11-08,0.525000,22.990000,2.283602,9.059941,trimestral
57,FUNO11,2025-02-07,0.551328,22.139999,2.490190,9.988124,trimestral
58,FUNO11,2025-05-08,0.554953,25.010000,2.218924,8.998971,trimestral
59,FUNO11,2025-08-08,0.570000,26.209999,2.174743,8.628055,trimestral
60,FUNO11,2025-11-07,0.605000,28.040001,2.157632,8.654238,trimestral
61,FUNO11,2026-02-06,0.670000,28.780001,2.328006,9.337604,trimestral
62,FUNO11,2026-05-08,0.620000,30.180000,2.054341,8.239938,trimestral
63,FUNO11,2026-08-07,0.639779,30.340000,2.108698,8.457965,trimestral


## Ficha de rendimiento anual

La ficha usa el **año calendario** (`1 de enero` a `31 de diciembre`). Los pagos se filtran por `ex_date`, que es la fecha disponible en el historial de `yfinance`; no se inventa una fecha de pago que la fuente no proporciona. Los precios inicial y final son el primer y último cierre disponible dentro del año. El rendimiento por dividendos se calcula contra el precio inicial, y el rendimiento de capital contra la variación entre precio final e inicial. La ficha es informativa y no constituye una recomendación de inversión.

In [7]:
AÑO_PRUEBA = 2025
ruta_ficha = crear_ficha_rendimiento(TICKER_PRUEBA, AÑO_PRUEBA, CARPETA_SALIDA, historial_dividendos)
print(f"Ficha generada: {ruta_ficha}")
display(HTML(ruta_ficha.read_text(encoding="utf-8")))

Ficha generada: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260822_184558_FUNO11_2025_ficha_rendimiento.html


ex_date,amount_mxn,yield_pct
2025-02-07,$0.5513,2.49%
2025-05-08,$0.5550,2.22%
2025-08-08,$0.5700,2.17%
2025-11-07,$0.6050,2.16%
